In [2]:
!pip install langchain langchain-community chromadb sentence-transformers



In [5]:
import os
import sys

# Fix path for notebook
sys.path.append(os.path.abspath("."))

# LangChain imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Tools
from tools.order_tool import get_order_status
from tools.product_tool import search_products

print("✅ Imports working")

✅ Imports working


In [6]:
# ----------------------------
# FILE PATH
# ----------------------------
BASE_DIR = os.getcwd()
FILE_PATH = os.path.join(BASE_DIR, "rag", "data.txt")

print("📂 Loading file from:", FILE_PATH)

# ----------------------------
# LOAD DOCUMENTS
# ----------------------------
loader = TextLoader(FILE_PATH)
documents = loader.load()

# ----------------------------
# SPLIT DOCUMENTS
# ----------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

📂 Loading file from: c:\Users\User\Desktop\milestone-2\mock-ecommerce-agent\rag\data.txt


In [7]:
# ----------------------------
# EMBEDDINGS
# ----------------------------
embeddings = HuggingFaceEmbeddings()

# ----------------------------
# VECTOR DB
# ----------------------------
DB_PATH = os.path.join(BASE_DIR, "db")

if os.path.exists(DB_PATH):
    print("⚡ Loading existing DB...")
    vectorstore = Chroma(
        persist_directory=DB_PATH,
        embedding_function=embeddings
    )
else:
    print("🧠 Creating new DB...")
    vectorstore = Chroma.from_documents(
        docs,
        embedding=embeddings,
        persist_directory=DB_PATH
    )

C:\Users\User\AppData\Local\Temp\ipykernel_11832\1338261799.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
C:\Users\User\AppData\Local\Temp\ipykernel_11832\1338261799.py:4: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3014.96it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------

🧠 Creating new DB...


In [14]:
# ----------------------------
# RETRIEVER
# ----------------------------
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

# ----------------------------
# RAG FUNCTION
# ----------------------------
def rag_search(query):

    # ✅ NEW METHOD
    relevant_docs = retriever.invoke(query)

    if not relevant_docs:
        return "Not found in knowledge base"

    context = relevant_docs[0].page_content[:200]

    return f"📚 From knowledge base:\n{context}"

In [15]:
def general_knowledge(query):

    q = query.lower()

    if "ai" in q:
        return "AI stands for Artificial Intelligence."

    elif "python" in q:
        return "Python is a programming language."

    return "I don't know the answer."

In [16]:
def route_question(question):

    q = question.lower()

    # 🔥 Order Tool
    if "order" in q:
        order_id = ''.join(filter(str.isdigit, question))
        return get_order_status(order_id)

    # 🔥 Product Tool
    elif any(word in q for word in ["laptop", "product"]):
        return search_products(question)

    # 🔥 RAG
    elif any(word in q for word in ["return", "policy", "shipping"]):
        return rag_search(question)

    # 🔥 Default
    else:
        return general_knowledge(question)

In [17]:
def ask_ai(question):
    return route_question(question)

In [18]:
ask_ai("Where is my order 1234?")


'Order 1234 is Shipped'

In [19]:
ask_ai("What is return policy?")



'📚 From knowledge base:\nReturn Policy:\nProducts can be returned within 7 days.\n\nShipping Policy:\nDelivery takes 3-5 business days.'

In [19]:
ask_ai("What is AI?")

'AI stands for Artificial Intelligence.'